# ComfyUI × Qwen Image 2.1
サイトで保存した設定を読み込み、実モデルを実行して結果JSONをダウンロードします。

「ランタイム → ランタイムのタイプを変更」でCUDA GPUを選んでください。無料T4での動作は未保証です。余裕のあるGPUメモリ・通常RAM・ディスクが必要です。

この開発環境ではGPU・有効キーでの実行は未検証です。失敗時はエラーで停止し、模擬結果を生成しません。依存関係は公式開発版を含み、環境によって調整が必要です。

Colabは対話的な実験用です。このノートブックは公開URL・トンネルを作りません。実行後はランタイムを停止してください。
[公式テンプレート](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_qwen_image_2_1_t2i.json)のノード構成を参考に、ローカルAPIで実行します。INT8版のDiT・エンコーダを使用します。生成も編集も対応。ComfyUIのWeb UIは公開しません。


## 1. インストール
パッケージを更新した後にランタイム再起動を求められた場合は、再起動し、次のセルから進んでください。

In [ ]:
%pip install huggingface_hub requests pillow
import subprocess, sys
from pathlib import Path
if not Path('/content/ComfyUI/.git').exists():
    subprocess.run(['git','clone','https://github.com/Comfy-Org/ComfyUI.git','/content/ComfyUI'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','/content/ComfyUI/requirements.txt'],check=True)
Path('/content/ComfyUI/input').mkdir(exist_ok=True)


## 2. 設定を読み込む
サイトから保存した設定JSONを1つ選びます。参照画像は後の実行セルで選びます。

In [ ]:
"""Shared helpers embedded into the Colab notebooks. No public server or tunnel."""
import base64
import datetime
import hashlib
import io
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

KINDS = ('comfyui', 'inference', 'prompt-rewrite', 'evals')

def validate_config(c, expected):
    if c.get('schema') != 'atlas-colab-config-v1' or c.get('kind') != expected:
        raise ValueError('別のデモの設定ファイルです。対象サイトから保存し直してください。')
    if not isinstance(c.get('prompt'), str) or not 1 <= len(c['prompt'].strip()) <= 4500:
        raise ValueError('プロンプトを確認してください。')
    for key, minimum, maximum in [('seed', 0, 2147483647), ('steps', 1, 50), ('repetitions', 1, 20)]:
        if type(c.get(key)) is not int or not minimum <= c[key] <= maximum:
            raise ValueError('設定値が不正です: ' + key)
    if c.get('size') not in (1024, 2048) or c.get('mode') not in ('generate', 'edit'):
        raise ValueError('サイズ・モードが不正です。')
    if not isinstance(c.get('transparent'), bool):
        raise ValueError('透過設定が不正です。')
    return c

def load_config(expected):
    from google.colab import files
    print('サイトから保存した atlas-' + expected + '-config.json を選択してください。')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('設定JSONは1件だけ選択してください。')
    raw = next(iter(uploaded.values()))
    if len(raw) > 20000:
        raise ValueError('設定ファイルが大きすぎます。')
    return validate_config(json.loads(raw), expected)

def gpu_info(required=True):
    try:
        result = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        if required:
            raise RuntimeError('CUDA GPUランタイムを選択してください。')
        result = 'CPU (API evaluation)'
    return {'gpu': result, 'python': platform.python_version()}

def effective_prompt(c):
    return ('This is an RGBA image with transparency. ' + c['prompt'] + ' The image has alpha channel and the background is transparent.') if c['transparent'] else c['prompt']

def upload_images(maximum=10):
    from google.colab import files
    from PIL import Image
    uploaded = files.upload()
    if not 1 <= len(uploaded) <= maximum:
        raise ValueError(f'画像は1〜{maximum}枚です。')
    images = []
    total = 0
    for data in uploaded.values():
        total += len(data)
        if len(data) > 4 * 1024**2 or total > 20 * 1024**2:
            raise ValueError('画像は1枚4MB、合計20MBまでです。')
        img = Image.open(io.BytesIO(data))
        if img.format not in ('PNG', 'JPEG', 'WEBP') or img.width * img.height > 20_000_000:
            raise ValueError('PNG/JPEG/WebP、2000万画素以下を選択してください。')
        img.load()
        images.append(img)
    return images

def image_data(image):
    output = io.BytesIO()
    image.save(output, format='PNG')
    if len(output.getvalue()) > 16 * 1024**2:
        raise ValueError('結果が16MBを超えました。1024pxで再実行してください。')
    return 'data:image/png;base64,' + base64.b64encode(output.getvalue()).decode('ascii')

def report(c, engine, model, elapsed, env, **extra):
    return {'schema': 'atlas-colab-result-v1', 'kind': c['kind'],
            'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'engine': engine, 'model': model, 'elapsed_seconds': elapsed,
            'environment': env, 'config': c, **extra}

def save_report(result):
    from google.colab import files
    path = Path('/content/atlas-' + result['kind'] + '-' + result['engine'] + '-result.json')
    path.write_text(json.dumps(result, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    files.download(str(path))
    print('結果JSONをサイトに読み込んでください。APIキーは結果に含めません。')

config=load_config('comfyui')
print(json.dumps(config,ensure_ascii=False,indent=2))
print(gpu_info(True))


## 3. 実モデルを実行
初回はモデルのダウンロードに時間がかかります。表示される画像・スコアは実行したモデルの結果です。

In [ ]:
import requests
from PIL import Image
from huggingface_hub import hf_hub_download
import uuid

def run_comfy(c):
    env = gpu_info()
    root = Path('/content/ComfyUI')
    root.mkdir(exist_ok=True)
    weights = ['diffusion_models/qwen_image_2.1_int8_convrot.safetensors',
               'text_encoders/qwen3vl_8b_int8_convrot.safetensors',
               'vae/qwen_image_2.1_vae_bf16.safetensors']
    for filename in weights:
        hf_hub_download('Comfy-Org/Qwen-Image-2.1', filename, local_dir=str(root / 'models'))
    env['revision'] = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    refs = upload_images() if c['mode'] == 'edit' else []
    prompt = effective_prompt(c)
    graph = {
      '1': {'class_type':'UNETLoader', 'inputs':{'unet_name':Path(weights[0]).name, 'weight_dtype':'default'}},
      '2': {'class_type':'CLIPLoader', 'inputs':{'clip_name':Path(weights[1]).name, 'type':'qwen_image', 'device':'default'}},
      '3': {'class_type':'VAELoader', 'inputs':{'vae_name':Path(weights[2]).name}},
      '4': {'class_type':'TextEncodeQwenImage21', 'inputs':{'clip':['2',0], 'vae':['3',0], 'prompt':prompt, 'negative_prompt':'', 'resolution':c['size']}},
      '5': {'class_type':'KSampler', 'inputs':{'model':['1',0], 'positive':['4',0], 'negative':['4',1], 'latent_image':['4',2], 'seed':c['seed'], 'steps':c['steps'], 'cfg':1.0, 'sampler_name':'euler', 'scheduler':'simple', 'denoise':1.0}},
      '6': {'class_type':'VAEDecode', 'inputs':{'samples':['5',0], 'vae':['3',0]}},
      '7': {'class_type':'SaveImage', 'inputs':{'images':['6',0], 'filename_prefix':'atlas_qwen'}}
    }
    ref_hashes = []
    for i, img in enumerate(refs, 1):
        name = 'atlas-' + uuid.uuid4().hex + '.png'
        img.save(root / 'input' / name)
        ref_hashes.append(hashlib.sha256((root / 'input' / name).read_bytes()).hexdigest())
        graph[str(10+i)] = {'class_type':'LoadImage', 'inputs':{'image':name}}
        graph['4']['inputs']['images.image_' + str(i)] = [str(10+i), 0]
    log_path = Path('/content/atlas-comfy.log')
    start = time.perf_counter()
    with log_path.open('w') as log:
        proc = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch'], cwd=root, stdout=log, stderr=subprocess.STDOUT)
        try:
            for _ in range(180):
                if proc.poll() is not None:
                    raise RuntimeError('ComfyUIが終了しました。/content/atlas-comfy.log を確認してください。')
                try:
                    info = requests.get('http://127.0.0.1:8188/object_info', timeout=5)
                    if info.ok:
                        assert 'TextEncodeQwenImage21' in info.json(), 'ComfyUIの対応ノードがありません。最新版で確認してください。'
                        break
                except requests.RequestException:
                    pass
                time.sleep(2)
            else:
                raise TimeoutError('ComfyUI起動がタイムアウトしました。')
            r = requests.post('http://127.0.0.1:8188/prompt', json={'prompt':graph, 'client_id':uuid.uuid4().hex}, timeout=30)
            if not r.ok:
                print(r.text[:3000])
                raise RuntimeError('ワークフローを実行できません。上のノードエラーを確認してください。')
            job_id = r.json()['prompt_id']
            for _ in range(1200):
                history = requests.get('http://127.0.0.1:8188/history/' + job_id, timeout=10).json().get(job_id)
                if history:
                    if history.get('status', {}).get('status_str') == 'error':
                        raise RuntimeError('GPU実行でエラー。/content/atlas-comfy.log を確認してください。')
                    outputs = history.get('outputs', {}).get('7', {}).get('images', [])
                    if outputs:
                        image_info = outputs[0]
                        content = requests.get('http://127.0.0.1:8188/view', params=image_info, timeout=60)
                        content.raise_for_status()
                        image = Image.open(io.BytesIO(content.content)); image.load()
                        elapsed = time.perf_counter() - start
                        display(image)
                        return report(c, 'comfyui', 'Comfy-Org/Qwen-Image-2.1 (INT8)', elapsed, env, image=image_data(image), timing_scope='startup_and_generation_excludes_download', reference_hashes=ref_hashes)
                time.sleep(2)
            raise TimeoutError('40分以内に結果を受信できませんでした。')
        finally:
            proc.terminate()
            try: proc.wait(timeout=20)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()

result=run_comfy(config)


## 4. 結果を保存してサイトへ戻る
結果JSONには入力文や生成物が含まれます。対象のデモで「結果JSONを読み込む」を選びます。サイトの読み込みはブラウザ内だけで処理します。

In [ ]:
save_report(result)
